# 06 — Covariance-controlled crossing with zero mean

Companion to the centered anisotropic Gaussian construction. It shows that second moments can carry slow/fast spectral information even when the ensemble mean remains exactly at equilibrium.


## Guiding question
Can two zero-mean Gaussian states relax differently solely because their covariance excites different dynamical sectors?


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm, solve_continuous_lyapunov
np.set_printoptions(precision=6, suppress=True)


Let the bath covariance be $\Sigma_b$ and choose a centered preparation
$$
\Sigma(0)=\Sigma_b+\Delta\Sigma(0).
$$
If $r_j$ is an eigenvector of $A$, the positive-semidefinite increment
$$
\Delta\Sigma_j(0)=a_j r_jr_j^T,
$$
evolves as $\Delta\Sigma_j(t)=a_jr_jr_j^Te^{2\lambda_jt}$. Its excess mean energy therefore decays with the same pure-mode rate as a displacement.


In [2]:
L=0.100; C=100e-6; R=110.
A=np.array([[0.,1.],[-1/(L*C),-R/L]])
G=np.diag([1/C,L])
w=np.linalg.eigvals(A).real; ls=max(w); lf=min(w)
rs=np.array([1.,ls]); rf=np.array([1.,lf])
def cov_increment_for_excess(r,E):
    return (2*E/(r@G@r))*np.outer(r,r)
Ds=cov_increment_for_excess(rs,1.0); Df=cov_increment_for_excess(rf,10.0)
assert np.all(np.linalg.eigvalsh(Ds)>-1e-12) and np.all(np.linalg.eigvalsh(Df)>-1e-12)
t=np.linspace(0,0.02,400); Es=[]; Ef=[]
for tt in t:
    P=expm(A*tt)
    S=P@Ds@P.T; F=P@Df@P.T
    Es.append(.5*np.trace(G@S)); Ef.append(.5*np.trace(G@F))
Es=np.array(Es); Ef=np.array(Ef)
tx=np.log(1/10)/(2*(lf-ls))
assert np.allclose(Es,np.exp(2*ls*t))
assert np.allclose(Ef,10*np.exp(2*lf*t))
print('covariance-only crossing [ms] =',1e3*tx)


covariance-only crossing [ms] = 1.2792139405522474


## Mixed covariance sector
In the slow--fast modal basis the covariance deviation contains three sectors:
$$
\sigma_{ss}\propto e^{2\lambda_st},\quad
\sigma_{sf}\propto e^{(\lambda_s+\lambda_f)t},\quad
\sigma_{ff}\propto e^{2\lambda_ft}.
$$
This is the second-moment analogue of the mixed-mode energy algebra.


In [3]:
Am=np.diag([ls,lf]); D0=np.array([[1.,0.4],[0.4,0.7]])
t2=np.linspace(0,0.02,200)
arr=[]
for tt in t2:
    P=expm(Am*tt); arr.append(P@D0@P.T)
arr=np.array(arr)
assert np.allclose(arr[:,0,0],D0[0,0]*np.exp(2*ls*t2))
assert np.allclose(arr[:,0,1],D0[0,1]*np.exp((ls+lf)*t2))
assert np.allclose(arr[:,1,1],D0[1,1]*np.exp(2*lf*t2))
print('three covariance sectors: PASS')


three covariance sectors: PASS


## Reader exploration
Rotate a positive covariance increment continuously between the slow and fast eigendirections. Plot the crossing time against the rotation angle and identify where a residual slow component dominates the late-time behavior.
